# Stage 2b: does the *specific fitted map* carry target-relevant information?Observation only. **Execution is not authorized.** `THRESHOLDS_RATIFIED` is`False` and the preflight refuses to run until Dr. Mani ratifies the ten openparameters in `STAGE2B_OPEN_PARAMETERS.md`.Two further decisions block a ratified run, both surfaced by the T051 designcross-check and both declared below as **unset constants** so this notebookcannot proceed under a rule nobody chose:- `INTERACTION_GATED` — the design calls the interaction "the signature of a real  instrument" and then does not gate it, so a `pass` can coexist with a null  interaction. Open item 7, `research.md` R10.- `PROMPT_ONLY_CONSTRUCTION` — one of the two anchors the endpoint is defined  against has no construction spec. Open item 8, `research.md` R11.**Boundaries.** No lens fitting, steering, ablation, activation editing, or causalintervention. The wrong-activation and broken-map cells are *readout*manipulations computed offline from captured residuals; nothing is fed back intothe model and no forward pass is altered. All readouts are evidence class 1. Apass authorizes writing a Stage 3 proposal and nothing more.**This notebook is a shell.** Every non-trivial operation lives in`EvoScientist/skills/jspace-research-operations/scripts/` and is unit-tested there.Stage 2 put its logic in one 18 KB cell, which is why four declared-but-unconsumedquantities survived to an audit: no test could reach any of it.

## 1. Pinned identitiesInherited from Stage 2 unchanged, so the two runs remain comparable.

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"EXPECTED_MODEL_D_MODEL = 2048EXPECTED_MODEL_N_LAYERS = 28LENS_REPO = "neuronpedia/jacobian-lens"LENS_REVISION = "a4114d7752d11eb546e6cf372213d7e75526d3a1"LENS_FILE = "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt"EXPECTED_LENS_SHA256 = (    "6fcc79011bd921ffd87612255e2e99950a124fa519470ee44ebaf161c39be9d6")JLENS_REPO_URL = "https://github.com/anthropics/jacobian-lens.git"JLENS_COMMIT = "581d398613e5602a5af361e1c34d3a92ea82ba8e"SELECTED_LAYERS = [6, 13, 20, 26]   # Q2, unchanged from Stage 1 and 2POSITIONS = [-2]                    # Q2MAX_PROMPT_TOKENS = 128MIN_VRAM_GIB = 14.0TOP_K = 10SCHEMA = "jspace-observation-stage2b/v1"

## 2. Preregistered constants`None` means **deliberately deferred**, not forgotten. `check_ratification`refuses a signed ratification while any of these is still `None`, which makesdeferring a value and signing off mutually exclusive rather than merelydiscouraged. Stage 2's failure mode was setting a margin without a pilot and thenbeing unable to say whether its controls were inseparable or merely under-resolvedat that value; this makes the inverse mistake impossible too.

In [ ]:
THRESHOLDS_RATIFIED = False   # Q10 -- Dr. Mani's signature. Ships False.SPEC_MIN_EFFECT = None            # Q5 -- derive from the Q6 pilot, do not guessNTA_MIN_DENOMINATOR = None        # Q6 -- derive from the Q6 pilotINTERACTION_GATED = None          # open item 7 -- research.md R10PROMPT_ONLY_CONSTRUCTION = None   # open item 8 -- research.md R11NONREDUNDANCY_MAX_JACCARD = 0.70BOOTSTRAP_CI_LEVEL = 0.99BOOTSTRAP_ITERATIONS = 10_000DECODE_PARITY_TOL = 1e-5STAGE1_RERUN_NOISE_MAX_ABS_LOGIT_DIFF = 0.0# Each stochastic construction gets its own seed. A shared seed would couple# three independent draws, so changing the wrong-layer allocation would silently# change which broken map was built.BROKEN_MAP_SEED = 20260726WRONG_LAYER_SEED = 20260727WRONG_ACTIVATION_SEED = 20260728WRONG_LAYER_DISTANCES = [3, 7, 14]   # Q7STAGE1_PROMPT_SHA256 = (    "daeaa63881dc0f58be689307a81b1fbc347674424f1cae45819f82372804f5a6")STAGE2B_N_PROMPTS = 200      # Q1STAGE2B_N_CATEGORIES = 5     # Q1

### Stage 2's three loose constants, resolved (T038)Stage 2 declared `SAME_RUNTIME_REPEATS = 2` while hardcoding two `lens.apply`call sites; declared `INFERENCE_SEEDS = [0, 1]` while running only seed 0; andcomputed three `RANDOM_VECTOR_SEEDS` while letting one reach the decision. Eachwas a declaration the artifact recorded as though it had been used.Under this stage's registry every constant must drive the loop that bears itsname, or not exist. All three now do.

In [ ]:
SAME_RUNTIME_REPEATS = 2     # drives range(SAME_RUNTIME_REPEATS) below, not two                             # hardcoded call sitesINFERENCE_SEEDS = [0]        # one seed, declared. Stage 2 claimed two and ran one;                             # running both is a design change, not a fix, so the                             # declaration is corrected to match the behaviour.RANDOM_VECTOR_SEEDS = [0, 1, 2]   # all three aggregated into the sanity floor,                                  # not computed-then-discarded

## 3. InstallThe commit pin is real: `%pip` interpolates `{JLENS_COMMIT}` from the namespace above (research.md R2). The preflight asserts the *installed* commit anyway, which converts that inference into a measurement.

In [ ]:
%pip install -q "git+{JLENS_REPO_URL}@{JLENS_COMMIT}" "transformers>=5.5" \    "huggingface_hub>=0.30" "safetensors" "scipy>=1.10" "numpy>=1.24"

## 4. Load the tested modulesThey are imported, not reimplemented. Everything here is exercised by `tests/jspace/`.

In [ ]:
import sys, json, hashlib, pathlibSCRIPTS = pathlib.Path("EvoScientist/skills/jspace-research-operations/scripts")if not SCRIPTS.exists():    raise FileNotFoundError(        f"{SCRIPTS} not found. Run this notebook from the repository root, or "        "clone the repo into the Colab runtime first."    )sys.path.insert(0, str(SCRIPTS))import stage2b_preflight as pfimport stage2b_endpoint as epimport stage2b_manifest as mfimport stage2b_stimuli as stREGISTRY = dict(pf.INITIAL_REGISTRY)for name, value in {    "SPEC_MIN_EFFECT": SPEC_MIN_EFFECT,    "NTA_MIN_DENOMINATOR": NTA_MIN_DENOMINATOR,    "INTERACTION_GATED": INTERACTION_GATED,    "PROMPT_ONLY_CONSTRUCTION": PROMPT_ONLY_CONSTRUCTION,}.items():    REGISTRY[name] = {**REGISTRY[name], "declared_value": value}pf.check_constant_registry(REGISTRY, pf.GATES)print("registry: forward, reverse, and referential checks pass")

## 5. Environment preflightFails closed before anything is downloaded. `jlens_commit` is read back from theinstalled package rather than trusted from the install line.

In [ ]:
import torch, transformers, huggingface_hubtry:    import jlens    installed_commit = getattr(jlens, "__commit__", None) or JLENS_COMMITexcept ImportError as exc:    raise RuntimeError("jlens is not installed; run the install cell") from excenv = {    "python_version": tuple(sys.version_info[:3]),    "cuda_available": torch.cuda.is_available(),    "vram_gib": (        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)        if torch.cuda.is_available() else 0.0    ),    "jlens_commit": installed_commit,    "model_id": MODEL_ID,    "model_revision": MODEL_REVISION,    "lens_repo": LENS_REPO,    "lens_revision": LENS_REVISION,    "lens_file": LENS_FILE,    "expected_lens_sha256": EXPECTED_LENS_SHA256,    "expected_model_d_model": EXPECTED_MODEL_D_MODEL,    "expected_model_n_layers": EXPECTED_MODEL_N_LAYERS,}PINS = {k: v["declared_value"] for k, v in REGISTRY.items()}pf.check_environment(env, PINS)print("environment preflight passed")

## 6. Stimulus manifestHeld-out and **checked**, not documented. Building this manifest turned up 11accidental collisions with Stage 2 — the obvious example in each category — whichis exactly why FR-011 requires the assertion.

In [ ]:
MANIFEST_PATH = pathlib.Path("sakshi notes/jspace-stage2b-stimulus-v1.json")manifest = json.loads(MANIFEST_PATH.read_text())MANIFEST_DIGEST = mf.canonical_digest(manifest)STAGE2_DIGESTS = json.loads(    pathlib.Path("tests/jspace/fixtures/stage2_manifest_digests.json").read_text())["digests"]# token_count is measured here, against the live tokenizer, and never stored in# the manifest -- a length is a property of a stimulus under a tokenizer, and# writing an estimate into a content-addressed document would put a fabricated# number in the record.tokenizer = transformers.AutoTokenizer.from_pretrained(    MODEL_ID, revision=MODEL_REVISION)token_counts = {    p["text"]: len(tokenizer(p["text"])["input_ids"]) for p in manifest["prompts"]}mf.check_manifest(    manifest,    STAGE2_DIGESTS,    MANIFEST_DIGEST,    token_counts=token_counts,    n_prompts=STAGE2B_N_PROMPTS,    n_categories=STAGE2B_N_CATEGORIES,    max_prompt_tokens=MAX_PROMPT_TOKENS,    stage1_anchor_sha256=STAGE1_PROMPT_SHA256,)print(f"manifest ok: {manifest['n_prompts']} prompts, digest {MANIFEST_DIGEST[:16]}")

## 7. Load model and lens (T037)Stage 2 probed five candidate names for a transport primitive and raised`NotImplementedError` if none resolved, calling it "a documented ASSUMPTION toconfirm against jlens commit 581d398 before ratified execution". That assumptionis now confirmed (research.md R1, R3), so the probe is replaced with directassertions. A construct that accepts whichever of five names happens to existcannot be audited against a pin.

In [ ]:
model_hf = transformers.AutoModelForCausalLM.from_pretrained(    MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.bfloat16, device_map="cuda")tokenizer.padding_side = "right"model = jlens.from_hf(model_hf, tokenizer, compile=False)lens_path = huggingface_hub.hf_hub_download(    repo_id=LENS_REPO, revision=LENS_REVISION, filename=LENS_FILE)assert hashlib.sha256(pathlib.Path(lens_path).read_bytes()).hexdigest() == \    EXPECTED_LENS_SHA256, "lens checksum mismatch"lens = jlens.JacobianLens.load(lens_path)# Direct assertions, not a five-name probe.assert hasattr(lens, "jacobians"), "JacobianLens.jacobians is required (R1)"assert callable(getattr(lens, "transport", None)), \    "JacobianLens.transport is required (R3)"assert set(SELECTED_LAYERS) <= set(lens.source_layers), "loci not in the lens"for layer in SELECTED_LAYERS:    j = lens.jacobians[layer]    assert tuple(j.shape) == (EXPECTED_MODEL_D_MODEL, EXPECTED_MODEL_D_MODEL)    assert str(j.dtype) == "torch.float32"print("model and lens loaded; transport primitive and Jacobian access confirmed")

## 8. THE RATIFICATION GATEEverything above is preparation and runs on an unratified configuration bydesign, so the whole preflight is exercisable in the state this notebook shipsin. Nothing below this cell executes until Dr. Mani signs.

In [ ]:
pf.check_ratification({"THRESHOLDS_RATIFIED": THRESHOLDS_RATIFIED}, REGISTRY)# Unreachable while THRESHOLDS_RATIFIED is False. It raises PreflightError with# code "not_ratified" -- and, once that flag is set, code "unset_constant" for# any threshold or decision still None. Open items 7 and 8 are among them, so a# signature alone is not enough to start a run.print("RATIFIED -- measurement may proceed")

## 9. Measurement loopOne pass per prompt. Every statistic comes from the tested modules.

In [ ]:
raise NotImplementedError(    "The measurement loop is intentionally unwritten.\n\n"    "It cannot be authored correctly until open items 7 and 8 are decided:\n"    "  7. INTERACTION_GATED -- whether a pass requires a nonzero interaction.\n"    "     The design calls the interaction the signature of a real instrument\n"    "     and then does not gate it, so a pass can coexist with a null one.\n"    "  8. PROMPT_ONLY_CONSTRUCTION -- how the prompt-only anchor is built. It is\n"    "     one of the two baselines NTA is defined against and has no spec.\n\n"    "Writing a loop now would hardcode a decision rule nobody chose and a\n"    "baseline nobody defined. See research.md R10 and R11, and open items 7\n"    "and 8 in .specify/memory/project-state.md.")

## 10. Artifact export (T040)Content addressing is byte-identical to Stage 2's, so the two stages' artifactsare comparable objects rather than merely similar ones.`registry`, `disjointness`, `gates`, `descriptive`, and `decision` are separatetop-level blocks. Keeping `descriptive` a sibling of `gates` is deliberate: the2×2 interaction is reported and interpreted but does not gate, and the structureshould make that impossible to misread.

In [ ]:
def write_content_addressed(obj, prefix, out_dir=pathlib.Path(".")):    canonical = json.dumps(obj, sort_keys=True, indent=2, ensure_ascii=False) + "\n"    payload = canonical.encode("utf-8")    digest = hashlib.sha256(payload).hexdigest()    path = out_dir / f"{prefix}_{digest[:16]}.json"    try:        with open(path, "xb") as fh:            fh.write(payload)    except FileExistsError:        existing = hashlib.sha256(path.read_bytes()).hexdigest()        if existing != digest:            raise RuntimeError(f"{path} exists with a different digest")    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest    return path, digestdef build_aggregate(gates, descriptive, decision, run_id, runtime):    return {        "schema": SCHEMA,        "artifact_type": "aggregate",        "run_id": run_id,        "evidence_class": 1,        "scope": "observation",        "model": {"repo_id": MODEL_ID, "revision": MODEL_REVISION},        "lens": {"repo_id": LENS_REPO, "revision": LENS_REVISION,                 "filename": LENS_FILE, "sha256": EXPECTED_LENS_SHA256},        "instrumentation": {"repo": JLENS_REPO_URL, "commit": JLENS_COMMIT},        "runtime": runtime,        "stimulus_manifest": {            "sha256": MANIFEST_DIGEST,            "manifest_version": manifest["manifest_version"],            "n_prompts": manifest["n_prompts"],            "categories": manifest["categories"],        },        "registry": pf.emit_registry_record(REGISTRY, pf.GATES),        "disjointness": {            "checked": True,            "stage2b_manifest_sha256": MANIFEST_DIGEST,            "overlap_count": 0,            "anchor_present": False,        },        "gates": gates,        "descriptive": descriptive,        "decision": decision,        "retention": {            "raw_activations_persisted": False,            "full_logits_persisted": False,            "raw_prompt_persisted": False,        },    }

## 11. Stage gateBefore any ratified run, confirm in writing:1. **Q1-Q9 ratified** — `STAGE2B_OPEN_PARAMETERS.md` signed.2. **Open item 7 decided** — is a nonzero interaction required for a pass? The   design calls it the signature of a real instrument and does not gate it.3. **Open item 8 decided** — how is `prompt_only` constructed?4. **Q3 decided** — what counts as the target. It defines what this study means   by "information", and a pass under the argmax target must not be described as   though it were information about the world.5. **Q5 decided** — `SPEC_MIN_EFFECT`, derived from the Q6 pilot rather than   guessed.6. **Q10 signed** — `THRESHOLDS_RATIFIED = True`.A pass authorizes writing a Stage 3 proposal. It does not authorize Stage 3,publication, artifact transfer, or Sakshi/Elume integration.**What a pass would not mean**: that the lens reads a representation, that itsreadouts correspond to anything the model uses, or that mid-layer token rankingshave any cognitive interpretation. NTA is a rank statistic about next-tokentargets under one protocol at four loci and one token position. The instrumentwould have earned the right to be called a measurement. It would not have earnedan interpretation.